# Experiment B2 — PPO + RND (Fixed Intrinsic Reward)

## Objective
This experiment evaluates whether Random Network Distillation (RND)
improves exploration performance over PPO-only baseline (B1).

## Configuration
- PPO backbone
- RND intrinsic reward
- Fixed reward weighting
- Shared encoder
- Dual critics:
  - extrinsic value critic
  - intrinsic value critic

## Expected Outcome
Compared to B1:
- Faster exploration
- Higher state coverage
- Better sparse reward discovery
- Improved FourRooms success rate

#### Imports


In [41]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA version: 11.8
Device count: 1
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU


In [42]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import yaml
import torch
from pathlib import Path

In [44]:
sys.path.append('C:/Users/rishe/minor_project_rl')
sys.path.append('C:/Users/rishe/minor_project_rl/scde')

In [45]:
from scde.utils.config import load_config
from scde.ppo.trainer import Trainer

#### Load the config

In [46]:
proj_root_path = 'C:/Users/rishe/minor_project_rl'

In [47]:
config_path = f'{proj_root_path}/scde/configs/b2_rnd_fixed.yaml'
cfg = load_config(config_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(cfg)

{'env_id': 'MiniGrid-FourRooms-v0', 'n_envs': 16, 'total_steps': 2000000, 'n_steps': 128, 'batch_size': 256, 'n_epochs': 4, 'gamma': 0.999, 'gamma_int': 0.99, 'gae_lambda': 0.95, 'clip_eps': 0.2, 'lr': 0.0003, 'entropy_coef': 0.01, 'value_coef': 0.5, 'max_grad_norm': 0.5, 'feature_dim': 256, 'action_dim': 7, 'use_rnd': True, 'rnd_dim': 512, 'rnd_update_proportion': 0.25, 'use_clip': False, 'clip_model': 'ViT-B-32', 'semantic_k': 5, 'buffer_size': 50000, 'norm_tau': 0.001, 'norm_clip': 5.0, 'adaptive': False, 'alpha': 1.0, 'beta': 0.5, 'alpha_max': 1.0, 'alpha_min': 0.05, 'beta_max': 0.5, 'beta_min': 0.01, 'tau_rho': 0.01, 'wandb': True, 'project_name': 'scde', 'run_name': 'B2_RND_fixed', 'defaults': ['base']}


In [48]:
print("use_rnd:", cfg.use_rnd)
print("use_clip:", cfg.use_clip)
print("adaptive:", cfg.adaptive)
print("device:", device)

use_rnd: True
use_clip: False
adaptive: False
device: cuda


### Intialize trainer and verify rnd exists

In [49]:
trainer = Trainer(cfg)

In [50]:
print(trainer.rnd)

RNDModule(
  (target): Sequential(
    (0): Linear(in_features=256, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
  )
  (predictor): Sequential(
    (0): Linear(in_features=256, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=512, bias=True)
  )
)


In [51]:
total_params = sum(
    p.numel()
    for p in trainer.optimizer.param_groups[0]["params"]
)

print("Optimizer params:", total_params)

Optimizer params: 1014857


#### sanity check forward pass

In [52]:
obs, _ = trainer.envs.reset()

obs = torch.tensor(
    obs,
    dtype=torch.uint8,
    device=trainer.device,
)

with torch.no_grad():

    action, log_prob, v_ext, v_int, h = (
        trainer.model.get_action(obs)
    )

print("action shape:", action.shape)
print("v_ext shape:", v_ext.shape)
print("v_int shape:", v_int.shape)
print("feature shape:", h.shape)

action shape: torch.Size([16])
v_ext shape: torch.Size([16])
v_int shape: torch.Size([16])
feature shape: torch.Size([16, 256])


#### Verify Intrisic Reward Computation

In [53]:
with torch.no_grad():

    r_int = trainer.rnd.intrinsic_reward(h)

print(r_int.shape)

print(r_int[:10])

torch.Size([16])
tensor([0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016,
        0.0016], device='cuda:0')


#### Verify RND Loss

In [54]:
rnd_loss = trainer.rnd.loss(h)

print(rnd_loss)

tensor(0.0016, device='cuda:0', grad_fn=<DivBackward0>)


#### Single Rollout Debug

In [55]:
trainer.buffer.reset()

obs, _ = trainer.envs.reset()

obs = torch.tensor(
    obs,
    dtype=torch.uint8,
    device=trainer.device,
)

with torch.no_grad():

    (
        action,
        log_prob,
        v_ext,
        v_int,
        h,
    ) = trainer.model.get_action(obs)

    r_int = trainer.rnd.intrinsic_reward(h)

print("Intrinsic reward:")
print(r_int[:10])

print()

print("Extrinsic values:")
print(v_ext[:10])

print()

print("Intrinsic values:")
print(v_int[:10])

Intrinsic reward:
tensor([0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016, 0.0016,
        0.0016], device='cuda:0')

Extrinsic values:
tensor([0.0266, 0.0242, 0.0262, 0.0257, 0.0246, 0.0220, 0.0247, 0.0249, 0.0264,
        0.0287], device='cuda:0')

Intrinsic values:
tensor([0.0242, 0.0246, 0.0242, 0.0222, 0.0221, 0.0257, 0.0274, 0.0229, 0.0238,
        0.0273], device='cuda:0')


#### Training smoke test

In [56]:
cfg.total_steps = 5000

trainer = Trainer(cfg)

trainer.train()

Steps: 2048 | Loss: -0.826 | Mean Return: 0.00 | Mean Ep Len: 100.0 | Success Rate: 0.00
Steps: 4096 | Loss: -0.840 | Mean Return: 0.00 | Mean Ep Len: 101.0 | Success Rate: 0.00
Steps: 6144 | Loss: -0.843 | Mean Return: 0.00 | Mean Ep Len: 101.0 | Success Rate: 0.02
Training completed.


#### Full Experiment

In [57]:
cfg.total_steps = 2000000

trainer = Trainer(cfg)

trainer.train()

Steps: 2048 | Loss: -0.709 | Mean Return: 0.09 | Mean Ep Len: 91.1 | Success Rate: 0.06
Steps: 4096 | Loss: -0.797 | Mean Return: 0.09 | Mean Ep Len: 91.7 | Success Rate: 0.06
Steps: 6144 | Loss: -0.875 | Mean Return: 0.05 | Mean Ep Len: 97.1 | Success Rate: 0.06
Steps: 8192 | Loss: -1.024 | Mean Return: 0.00 | Mean Ep Len: 101.0 | Success Rate: 0.05
Steps: 10240 | Loss: -0.777 | Mean Return: 0.10 | Mean Ep Len: 91.4 | Success Rate: 0.05
Steps: 12288 | Loss: -0.924 | Mean Return: 0.00 | Mean Ep Len: 101.0 | Success Rate: 0.06
Steps: 14336 | Loss: -0.809 | Mean Return: 0.00 | Mean Ep Len: 101.0 | Success Rate: 0.05
Steps: 16384 | Loss: -0.808 | Mean Return: 0.00 | Mean Ep Len: 101.0 | Success Rate: 0.03
Steps: 18432 | Loss: -1.079 | Mean Return: 0.00 | Mean Ep Len: 101.0 | Success Rate: 0.03
Steps: 20480 | Loss: -0.844 | Mean Return: 0.00 | Mean Ep Len: 101.0 | Success Rate: 0.03
Steps: 22528 | Loss: -0.775 | Mean Return: 0.07 | Mean Ep Len: 94.1 | Success Rate: 0.02
Steps: 24576 | Loss

#### Plots

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(all_returns)
plt.title("Episodic Return")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.show()

NameError: name 'all_returns' is not defined

<Figure size 1000x500 with 0 Axes>

: 

## Post Training notes
### Observations

Record:

- exploration quality
- sparse reward discoveries
- episode returns
- stability
- intrinsic reward behaviour
- comparison against B1

Questions:
1. Does RND improve exploration?
2. Does intrinsic reward decay over time?
3. Is exploration more diverse?
4. Does PPO remain stable?

### Save model

In [ ]:
SAVE_DIR = Path(f'{proj_root_path}/checkpoints')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
torch.save(
    trainer.model.state_dict(),
    SAVE_DIR / "b2_model.pt",
)

if trainer.rnd is not None:

    torch.save(
        trainer.rnd.state_dict(),
        SAVE_DIR / "b2_rnd.pt",
    )

print("Saved.")

Saved.


#### Load model check

In [ ]:
model_path = SAVE_DIR / "b2_model.pt"

state_dict = torch.load(
    model_path,
    map_location=trainer.device,
)

trainer.model.load_state_dict(state_dict)

print("Loaded successfully.")

Loaded successfully.


#### Experimentation summary

# Experiment B2 Summary

## Features Enabled
- PPO
- RND intrinsic reward
- Dual critics
- Shared encoder

## Features Disabled
- CLIP semantic rewards
- Adaptive weighting

## Goal
Evaluate whether curiosity-driven exploration improves
performance in sparse-reward environments.